# CLIMES combined candidate — October 19

In Colab select **Runtime → Change runtime type → T4 GPU**. Run the two code cells in order. The first prepares a separate project directory; the second runs checks, the live 17-case batch, downloads its evidence ZIP, and starts the UI. Do not click Merge until the live results and example answers have been reviewed.

离线检查已经完成。真实模型、资料检索、天气 API 和界面需要在这里验证。运行后保留 Colab 连接；`gradio.live` 链接依赖正在运行的服务。

In [ ]:
from pathlib import Path
import subprocess, tempfile
base = Path(tempfile.mkdtemp(prefix="climes_repository_", dir="/content"))
project = base/"LLMgroup2"
subprocess.run(["git","clone","--depth","1","https://github.com/lkjidm/LLMgroup2.git",str(project)],check=True)
if not (project/"climes_current/check_integration.py").is_file():
    raise RuntimeError("The combined candidate is not merged into main yet. Use the candidate ZIP notebook to test it before merging.")
print("Git commit:", subprocess.check_output(["git","rev-parse","HEAD"],cwd=project,text=True).strip())

import sys, subprocess, importlib.metadata
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun this cell.")
print("GPU:", torch.cuda.get_device_name(0))
expected = {"transformers":"5.19.0", "accelerate":"1.15.0", "gradio":"6.29.0",
            "langchain-community":"0.4.2", "chromadb":"1.5.9",
            "sentence-transformers":"5.7.0", "requests":"2.34.2", "edge-tts":"7.3.1"}
def needs_install():
    try:
        return any(importlib.metadata.version(name) != value for name, value in expected.items())
    except importlib.metadata.PackageNotFoundError:
        return True
if needs_install():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(project/"requirements.txt")], check=True)
print("Prepared:", project)
print("Next: run the second cell.")


In [ ]:
import os, sys, getpass
if not os.environ.get("WEATHERMETRO_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["WEATHERMETRO_API_KEY"] = userdata.get("WEATHERMETRO_API_KEY") or ""
    except Exception:
        pass
if not os.environ.get("WEATHERMETRO_API_KEY"):
    os.environ["WEATHERMETRO_API_KEY"] = getpass.getpass("WeatherMetro API key (hidden): ").strip()
if not os.environ.get("WEATHERMETRO_API_KEY"):
    raise RuntimeError("No API key supplied. Add the Colab Secret WEATHERMETRO_API_KEY, then rerun this cell.")
os.chdir(project)
sys.path.insert(0, str(project))
import runpy
runner = runpy.run_path(str(project/"climes_current/START.py"), run_name="climes_combined_acceptance")
report, evidence_zip = runner["main"](package=project/"climes_current", project=project,
                                       output_root=project.parent, open_ui=True, download=True)
